# Fixed Training Notebook

This notebook is a refactored, safe, and reproducible **training** notebook created from your original `Copy_of_finals_clean.ipynb`. It includes:

- package/version logging
- clear data loading cell
- EDA placeholders
- preprocessing pipeline (with automatic numeric/categorical detection)
- train/test split and saving of test set
- model pipelines (example: Ridge and RandomForest) wrapped in `Pipeline`
- hyperparameter search with `GridSearchCV` / `RandomizedSearchCV`
- saving of best pipeline(s) + metadata
- basic training diagnostics and CV results

> Edit the file paths and `TARGET_COLUMN` variable to match your dataset.

---

Created: 2025-10-09T21:21:01.980632Z


In [ ]:
# Cell 1: Environment and versions
import sys, platform
import sklearn, pandas as pd, numpy as np
print("python:", sys.version.splitlines()[0])
print("platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
# Cell 2: Imports
import os
import joblib
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Cell 3: Configuration - edit these paths/names to match your project
DATA_PATH = 'data/dataset.csv'  # <-- change to your dataset path (or mount your Google Drive)
TARGET_COLUMN = 'target'        # <-- change to your target column name
OUTPUT_DIR = 'trained_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Helpful: print current working dir
print("CWD:", os.getcwd())
print("Will save artifacts to:", OUTPUT_DIR)

In [ ]:
# Cell 4: Load dataset
# Replace DATA_PATH if needed. The cell attempts to read CSV, but you can replace with other loaders.
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}. Update DATA_PATH in Cell 3.")

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
df.head()

## Cell 5: Quick EDA

Run the following code to inspect missing values, dtypes, and a simple distribution check. Modify as needed.

In [ ]:
# Cell 6: EDA - missing values and datatypes
print('Columns and dtypes:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum().sort_values(ascending=False).head(20))

# Basic numeric description
print('\nNumeric summary:')
display(df.select_dtypes(include=[np.number]).describe().T)

In [ ]:
# Cell 7: Define X and y, quick checks
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found in dataset columns: {df.columns.tolist()}")

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)

# Basic check: if y has missing values
print('Missing in target:', y.isna().sum())

In [ ]:
# Cell 8: Auto-detect numeric and categorical columns
numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('Numeric columns:', len(numeric_cols))
print('Categorical columns:', len(categorical_cols))

# If you want to override:
# numeric_cols = ['col1', 'col2', ...]
# categorical_cols = ['cat1', 'cat2', ...]

In [ ]:
# Cell 9: Train/test split (save test set for separate testing notebook)
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print('X_train:', X_train.shape, 'X_test:', X_test.shape)

# Save test set so testing notebook can load it independently
X_test.to_csv(os.path.join(OUTPUT_DIR, 'X_test.csv'), index=False)
y_test.to_csv(os.path.join(OUTPUT_DIR, 'y_test.csv'), index=False)
print('Saved X_test and y_test to', OUTPUT_DIR)

In [ ]:
# Cell 10: Preprocessing pipeline
# Numeric pipeline: impute (median) + scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute (most_frequent) + one-hot encode (drop='first' to avoid collinearity)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
], remainder='drop')  # drop any other columns

print('Preprocessor created with %d numeric and %d categorical columns.' % (len(numeric_cols), len(categorical_cols)))

In [ ]:
# Cell 11: Define pipelines for candidate models and search spaces

# Ridge pipeline
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(random_state=RANDOM_STATE))
])

ridge_param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

# Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_param_dist = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10]
}

print('Pipelines defined')

In [ ]:
# Cell 12: Hyperparameter search (Ridge GridSearchCV & RandomizedSearchCV for RandomForest)
from sklearn.model_selection import GridSearchCV

# Ridge (fast)
print('Running GridSearchCV for Ridge...')
ridge_search = GridSearchCV(ridge_pipeline, ridge_param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_search.fit(X_train, y_train)
print('Ridge best params:', ridge_search.best_params_)
print('Ridge best CV score (neg MAE):', ridge_search.best_score_)

# RandomForest (randomized search)
print('\nRunning RandomizedSearchCV for RandomForest...')
rf_search = RandomizedSearchCV(rf_pipeline, rf_param_dist, n_iter=6, cv=3,
                               scoring='neg_mean_absolute_error', n_jobs=-1, random_state=RANDOM_STATE)
rf_search.fit(X_train, y_train)
print('RF best params:', rf_search.best_params_)
print('RF best CV score (neg MAE):', rf_search.best_score_)

In [ ]:
# Cell 13: Select best model by CV score and refit on full training data
# We compare the best CV (neg MAE) from both
candidates = {
    'ridge': (ridge_search.best_estimator_, ridge_search.best_score_),
    'random_forest': (rf_search.best_estimator_, rf_search.best_score_)
}

best_name, (best_pipeline, best_score) = max(candidates.items(), key=lambda kv: kv[1][1])
print(f"Selected best model: {best_name} with CV score (neg MAE) = {best_score}")

# Refit best pipeline on the entire training set (GridSearchCV/RandomizedSearchCV already refit by default on sklearn>=0.24)
best_pipeline.fit(X_train, y_train)

# Save pipeline
timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
model_filename = os.path.join(OUTPUT_DIR, f"best_pipeline_{best_name}_{timestamp}.joblib")
joblib.dump(best_pipeline, model_filename)
print('Saved best pipeline to', model_filename)

In [ ]:
# Cell 14: Save training metadata (features, versions, cv scores)
metadata = {
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'model_name': best_name,
    'model_filename': os.path.basename(model_filename),
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'ridge_cv_neg_mae': float(ridge_search.best_score_),
    'rf_cv_neg_mae': float(rf_search.best_score_),
    'selected_cv_neg_mae': float(best_score),
    'sklearn_version': sklearn.__version__
}

with open(os.path.join(OUTPUT_DIR, 'training_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved metadata to', os.path.join(OUTPUT_DIR, 'training_metadata.json'))

In [ ]:
# Cell 15: Evaluate the saved best_pipeline on the held-out test set
# Load test data saved earlier (ensures evaluation notebook can be separate)
X_test = pd.read_csv(os.path.join(OUTPUT_DIR, 'X_test.csv'))
y_test = pd.read_csv(os.path.join(OUTPUT_DIR, 'y_test.csv')).iloc[:, 0]  # assumes single-column

# Predict
y_pred = best_pipeline.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Test set MAE: {mae:.4f}')
print(f'Test set MSE: {mse:.4f}')
print(f'Test set R2: {r2:.4f}')

# Save predictions
pred_df = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred})
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'test_set_predictions.csv'), index=False)
print('Saved predictions to', os.path.join(OUTPUT_DIR, 'test_set_predictions.csv'))

In [ ]:
# Cell 16: Diagnostic plots - Predicted vs Actual and Residuals
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual')
plt.show()

# Residuals
residuals = y_test - y_pred
plt.figure(figsize=(8,6))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='k', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted')
plt.show()

In [ ]:
# Cell 17: Feature importance (tree-based) or coefficients (linear)
def get_feature_names(preprocessor):
    output_names = []
    # numeric names (pass-through)
    output_names.extend(numeric_cols)
    # categorical names from onehot encoder
    if 'cat' in preprocessor.named_transformers_:
        ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
        if hasattr(ohe, 'get_feature_names_out'):
            cat_names = list(ohe.get_feature_names_out(categorical_cols))
            output_names.extend(cat_names)
    return output_names

try:
    preproc = best_pipeline.named_steps['preprocessor']
    feat_names = get_feature_names(preproc)
    model = best_pipeline.named_steps['model']
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        fi = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(30)
        display(fi)
    elif hasattr(model, 'coef_'):
        coefs = model.coef_
        coef_series = pd.Series(coefs, index=feat_names).sort_values(key=abs, ascending=False).head(30)
        display(coef_series)
    else:
        print('Model has no feature_importances_ or coef_ attribute to show.')
except Exception as e:
    print('Could not compute feature importances or coefficients:', e)

## Done

This notebook saved:

- Best pipeline joblib file (in `trained_models/`)
- `training_metadata.json`
- `X_test.csv`, `y_test.csv`, `test_set_predictions.csv`

Next steps:

- Inspect `training_metadata.json` for reproducibility details.
- Create a separate **testing notebook** that loads the saved pipeline and runs evaluation/plots on `trained_models/X_test.csv` / `y_test.csv`.

If you want, I can also generate that testing notebook now.